In [2]:
import random as rd
from tqdm import tqdm
from typing import Tuple, Optional
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import f1_score

import numpy as np
import pandas as pd

from datasets import load_dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.normalizers import Lowercase
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.decoders import BPEDecoder, Replace

from transformers import PreTrainedTokenizerFast

c:\main\GitHub\nlpTasks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
dataset = load_dataset("mteb/emotion")
print(f"Dataset type: {type(dataset)}")
dataset

Dataset type: <class 'datasets.dataset_dict.DatasetDict'>


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 15956
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1988
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1986
    })
})

In [4]:
train_dataset = dataset["train"].to_pandas()
val_dataset = dataset["validation"].to_pandas()
test_dataset = dataset['test'].to_pandas()
print(f"Splits dtype: {type(train_dataset), type(val_dataset), type(train_dataset)}")
print(len(train_dataset), len(val_dataset), len(test_dataset))

Splits dtype: (<class 'pandas.DataFrame'>, <class 'pandas.DataFrame'>, <class 'pandas.DataFrame'>)
15956 1988 1986


In [5]:
print("Random train row:")
print(train_dataset.sample(), end = '\n\n')
print("Random val row:")
print(val_dataset.sample(), end = '\n\n')
print("Random test row:")
print(test_dataset.sample())

Random train row:
                                                  text  label label_text
882  i feel like an innocent victim i feel that i j...      1        joy

Random val row:
                                                  text  label label_text
726  i feel for matters at hand to be resolved thes...      1        joy

Random test row:
                                                   text  label label_text
1774  im feeling all jolly and warm inside but i jus...      1        joy


In [78]:
# Unique labels
train_dataset[["label", "label_text"]].drop_duplicates()

,label,label_text
0,0,sadness
2,3,anger
3,2,love
6,5,surprise
7,4,fear
8,1,joy


In [32]:
def get_label_text(label: int) -> str:
    label_data = train_dataset[["label", "label_text"]].drop_duplicates()
    for row in label_data.itertuples():
        if row.label == label:
            return row.label_text
    raise ValueError("There is no such label")

In [237]:
# Create tokeinzer object from HF tokenizers
tokenizer = Tokenizer(model = BPE(unk_token="[UNK]"))

# Trainer tokenizer
trainer = BpeTrainer(
    vocab_size =200,  # Desired vocabulary size
    min_frequency= 1,   # Minimum frequency for a token to be included
    special_tokens=["[PAD]", "[UNK]", "[CLS]"]
)
tokenizer.normalizer = Lowercase()
tokenizer.decoder = BPEDecoder(suffix = "#")

In [238]:
train_dataset["text"][:5]

0                              i didnt feel humiliated
1    i can go from feeling so hopeless to so damned...
2     im grabbing a minute to post i feel greedy wrong
3    i am ever feeling nostalgic about the fireplac...
4                                 i am feeling grouchy
Name: text, dtype: str

In [239]:
tokenizer.train_from_iterator(train_dataset["text"], trainer = trainer)

In [240]:
vocab = tokenizer.get_vocab()
vocab

{'x': 27,
 'op': 176,
 'sel': 191,
 'ly ': 73,
 'er ': 67,
 'su': 126,
 'le': 88,
 'is ': 74,
 't ': 31,
 'all ': 169,
 'like ': 98,
 'sh': 106,
 'ough': 185,
 'when ': 170,
 'ex': 197,
 'u': 24,
 'it ': 87,
 'ro': 119,
 'be ': 134,
 'di': 110,
 'because ': 195,
 're': 56,
 'pe': 145,
 'that ': 70,
 'es': 105,
 'wa': 83,
 'en': 48,
 'ke ': 84,
 'you': 127,
 'wor': 189,
 'cause ': 188,
 'i c': 158,
 'er': 45,
 'feel': 41,
 'som': 137,
 'of ': 71,
 'to': 164,
 ' ': 3,
 'ab': 94,
 'da': 148,
 'ent': 182,
 'ch': 86,
 'not ': 136,
 'n ': 190,
 'com': 152,
 'r': 21,
 'an': 39,
 'ti': 63,
 'ow': 79,
 'al ': 181,
 'ver': 159,
 'fu': 142,
 'd': 7,
 'p ': 117,
 'li': 64,
 'o ': 43,
 'g': 10,
 'si': 135,
 'de': 162,
 'n': 17,
 'om': 75,
 'ould ': 160,
 'al': 55,
 'just ': 165,
 'con': 180,
 'ev': 116,
 'tt': 122,
 'me ': 95,
 'more ': 199,
 'i am ': 133,
 'ha': 65,
 'th': 32,
 'ch ': 146,
 'with ': 130,
 'now ': 174,
 'ac': 108,
 'in': 33,
 'so ': 100,
 'the ': 54,
 'z': 29,
 'ir': 138,
 'i ': 35

In [241]:
# CLS token
vocab["[CLS]"]

2

In [242]:
main_tokenizer = PreTrainedTokenizerFast(tokenizer_object = tokenizer, pad_token = "[PAD]")

In [243]:
# Encoding with cls token
input_ids = main_tokenizer("[CLS]" + "never you mind that weakling", padding = "max_length", max_length = 50, trucation = True)['input_ids']
input_ids

[2,
 17,
 116,
 67,
 167,
 16,
 33,
 34,
 70,
 26,
 8,
 4,
 14,
 15,
 42,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [244]:
max_number_of_tokens = 0
for text in train_dataset["text"]:
    tokenized_text = main_tokenizer(text)['input_ids']
    max_number_of_tokens = max(max_number_of_tokens, len(tokenized_text))
print(f"Maxium number of tokens in a single sentence: {max_number_of_tokens}")

Maxium number of tokens in a single sentence: 160


In [245]:
main_tokenizer.decode(input_ids, skip_special_tokens = True)

'never you mind that weakling'

In [246]:
# Create torch datasets
class EmotionDataset(Dataset):
    def __init__(self, data: pd.DataFrame, 
                 tokenizer: PreTrainedTokenizerFast, max_length: int = 108):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __getitem__(self, idx) -> Tuple[torch.tensor, int]:
        single_row = self.data.iloc[idx]
        raw_text = single_row["text"]
        tokenized_text = self.tokenizer("[CLS]" + raw_text, padding = "max_length",
                                        max_length = self.max_length, truncation = True,
                                        return_tensors = "pt")["input_ids"]
        return tokenized_text.squeeze(dim = 0), single_row["label"].item()

    def __len__(self):
        return len(self.data)


In [247]:
# Random emotions dataset item
emotion_dataset = EmotionDataset(train_dataset, tokenizer = main_tokenizer)
random_item = emotion_dataset[rd.randint(0, len(emotion_dataset) - 1)]
print(f"Random item tensor shape: {random_item[0].shape}")
random_item

Random item tensor shape: torch.Size([108])


(tensor([  2, 133,  68,  60,  23,  44,  53,  49,  98,  62, 189,  32,  88,  22,
          36,  19,  45,  22,  47,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
           0,   0,   0,   0,   0,   0,   0,   0,   0,   0]),
 0)

In [248]:
# Prepare all the other datasets
emotions_dataset_train = EmotionDataset(train_dataset, tokenizer = main_tokenizer)
emotions_dataset_val = EmotionDataset(val_dataset, tokenizer = main_tokenizer)
emotions_dataset_test = EmotionDataset(test_dataset, tokenizer = main_tokenizer)
print(f"Lengths of datasets: {len(emotions_dataset_train)}, {len(emotions_dataset_val)}, {len(emotions_dataset_test)}")


Lengths of datasets: 15956, 1988, 1986


In [249]:
# Create emotion model
class EmotionModel(nn.Module):
    def __init__(self, input_size: int, hidden_size: int,
                 embedding_num: int, pad_id: int = 0, num_layers: int = 1, num_cls: int = 6):
        super().__init__()
        self.embed = nn.Embedding(embedding_num, input_size, padding_idx = pad_id)
        self.linear = nn.Linear(hidden_size, num_cls)
        self.rnn = nn.RNN(input_size, hidden_size, batch_first = False, num_layers = num_layers)
    
    def forward(self, x):
        y = self.embed(x)
        y = y.transpose(0, 1)
        out, h = self.rnn(y)
        y = out[0].squeeze(dim = 0)
        y = self.linear(y)
        return y

In [250]:
def inference(text: str, model: nn.Module) -> Tuple[int, str]:
    pass

In [265]:
@dataclass
class TrainConfig:
    batch_size: int = 32
    learning_rate: float = 5e-2
    betas: Tuple[float, float] = (0.99, 0.98)
    epoch: int = 100

In [266]:
config = TrainConfig()
config

TrainConfig(batch_size=32, learning_rate=0.05, betas=(0.99, 0.98), epoch=100)

In [253]:
train_dataloader = DataLoader(dataset = emotions_dataset_train,
                              shuffle = True, batch_size = config.batch_size)

In [254]:
val_loader = DataLoader(dataset = emotions_dataset_val, shuffle = False, batch_size = config.batch_size)
val_loader

In [255]:
train_sample = next(iter(train_dataloader))
print(f"Shape tensor of train sample: {train_sample[0].shape}")

Shape tensor of train sample: torch.Size([32, 108])


In [256]:
len(main_tokenizer.get_vocab())

200

In [257]:
rnn_model = EmotionModel(input_size = 300,
                        hidden_size = 300,
                        embedding_num = len(main_tokenizer.get_vocab())).to(device = 'cuda')
rnn_model

EmotionModel(
  (embed): Embedding(200, 300, padding_idx=0)
  (linear): Linear(in_features=300, out_features=6, bias=True)
  (rnn): RNN(300, 300)
)

In [258]:
rnn_model(train_sample[0].to(device = 'cuda')).shape

torch.Size([32, 6])

In [259]:
# Training loop for model_training
def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    """Run one training epoch."""
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for inputs, targets in tqdm(dataloader, desc="Epoch training"):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [260]:
example_input = emotions_dataset_train.tokenizer("The waiters at this restaurant are awful", return_tensors = "pt")["input_ids"]

In [261]:
# Given the model calculate f1-score on validation dataset
def calculate_val_score(dataloader: DataLoader,
                        model: nn.Module, device: str = "cuda") -> float:
    model.eval()
    y_preds = []
    y_trues = []
    for input_tensors, labels in dataloader:
        input_tensors, labels = input_tensors.to(device = device), labels.to(device = device)
        predictions = model(input_tensors)
        y_pred = predictions.softmax(dim = 1).argmax(dim = 1).detach().tolist()
        y_true = labels.detach().tolist()
        y_preds.extend(y_pred)
        y_trues.extend(y_true)
    return f1_score(y_true = y_trues, y_pred = y_preds, average = "macro")

In [267]:
rnn_model = EmotionModel(input_size = 300,
                         hidden_size = 300,
                         embedding_num = len(main_tokenizer.get_vocab())).to(device = 'cuda')

In [268]:
optimizer = optim.SGD(params = rnn_model.parameters(), lr = config.learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index = 0)

In [ ]:
# Training looop 
for ep in tqdm(range(config.epoch), desc = "Current epoch"):
    current_loss = train_one_epoch(model = rnn_model,
                    dataloader = train_dataloader,
                    optimizer = optimizer,
                    criterion = criterion,
                    device = 'cuda')
    current_val = calculate_val_score(val_loader, rnn_model, device = "cuda")
    print(f"Epoch # {ep + 1} | Train loss: {current_loss} | Val f1_score: {current_val}")
    

Current epoch:   1%|          | 1/100 [00:10<18:05, 10.97s/it]

Epoch # 1 | Train loss: 1.4033745639548751 | Val f1_score: 0.08680555555555557


Current epoch:   2%|▏         | 2/100 [00:21<17:29, 10.71s/it]

Epoch # 2 | Train loss: 1.3842247932372924 | Val f1_score: 0.08680555555555557


Current epoch:   3%|▎         | 3/100 [00:32<17:18, 10.70s/it]

...

Epoch # 11 | Train loss: 1.3767341578412868 | Val f1_score: 0.08680555555555557


Current epoch:  12%|█▏        | 12/100 [02:07<15:39, 10.67s/it]

Epoch # 12 | Train loss: 1.3767415596870238 | Val f1_score: 0.08680555555555557


Current epoch:  13%|█▎        | 13/100 [02:18<15:27, 10.66s/it]

Epoch # 13 | Train loss: 1.3741973120129418 | Val f1_score: 0.08680555555555557
